# Scaffolds and Butina: clustering molecules and splitting datasets

Selected FreeSolv v0.52 subset; experimental hydration free energy, kcal/mol; data credited to Mobley and colleagues, **CC BY 4.0**.

[Original data (v0.52)](https://github.com/MobleyLab/FreeSolv/blob/v0.52/database.txt) · [Dataset release](https://doi.org/10.5281/zenodo.1161245) · [Data license](https://github.com/MobleyLab/FreeSolv/blob/master/LICENSE)

The labels describe gas-to-water hydration free energy and are not used for grouping. Molecule names are comments only. The two clustering methods are separate experiments.


## Installation
Run in your terminal or active notebook environment:
```bash
python -m pip install rdkit pandas
```
This saved notebook was executed with RDKit 2025.09.6 and pandas 2.2.2. Install those versions to reproduce the recorded outputs exactly:
```bash
python -m pip install rdkit==2025.9.6 pandas==2.2.2
```


In [1]:
import pandas as pd
from rdkit import Chem


In [2]:
# FreeSolv v0.52 subset; data: CC BY 4.0
# Experimental hydration free energies in kcal/mol.
rows = [
    ("c1ccccc1", -0.90),             # benzene
    ("Cc1ccccc1", -0.90),            # toluene
    ("CCc1ccccc1", -0.79),           # ethylbenzene
    ("c1ccc(cc1)O", -6.60),          # phenol
    ("COc1ccccc1", -2.45),           # anisole
    ("c1ccc(cc1)Cl", -1.12),         # chlorobenzene
    ("c1ccncc1", -4.69),             # pyridine
    ("Cc1ccccn1", -4.63),            # 2-methylpyridine
    ("Cc1cccnc1", -4.77),            # 3-methylpyridine
    ("Cc1ccncc1", -4.93),            # 4-methylpyridine
    ("C1CCCCC1", 1.23),              # cyclohexane
    ("CC1CCCCC1", 1.70),             # methylcyclohexane
    ("C1CCC(CC1)O", -5.46),          # cyclohexanol
    ("C1CCC(CC1)N", -4.59),          # cyclohexanamine
    ("c1ccc2ccccc2c1", -2.40),       # naphthalene
    ("Cc1cccc2c1cccc2", -2.44),      # 1-methylnaphthalene
    ("c1ccsc1", -1.40),              # thiophene
    ("Cc1cccs1", -1.38),             # 2-methylthiophene
    ("CCO", -5.00),                  # ethanol
    ("CCCO", -4.85),                 # propan-1-ol
]

In [3]:
data = pd.DataFrame(rows, columns=["smiles", "label"])
mols = [Chem.MolFromSmiles(s) for s in data["smiles"]]
assert all(mol is not None for mol in mols)
data


,smiles,label
0,c1ccccc1,-0.90
1,Cc1ccccc1,-0.90
2,CCc1ccccc1,-0.79
3,c1ccc(cc1)O,-6.60
4,COc1ccccc1,-2.45
5,c1ccc(cc1)Cl,-1.12
6,c1ccncc1,-4.69
7,Cc1ccccn1,-4.63
8,Cc1cccnc1,-4.77
9,Cc1ccncc1,-4.93


## Scaffold grouping
The key is a canonical scaffold SMILES. `i` is the zero-based original row position. The empty string groups ring-free molecules.

In [4]:
from rdkit.Chem.Scaffolds import MurckoScaffold

scaffold_groups = {}
for i, mol in enumerate(mols):
    key = MurckoScaffold.MurckoScaffoldSmiles(
        mol=mol, includeChirality=False
    )
    scaffold_groups.setdefault(key, []).append(i)

scaffold_clusters = list(scaffold_groups.values())
print("Scaffold clusters:", scaffold_clusters)


Scaffold clusters: [[0, 1, 2, 3, 4, 5], [6, 7, 8, 9], [10, 11, 12, 13], [14, 15], [16, 17], [18, 19]]


## Morgan fingerprints
Binary Morgan fingerprints, radius 2, 2,048 bits, no chirality. These describe complete molecules, including side chains.

In [5]:
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator

fp_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2, fpSize=2048, includeChirality=False
)
fps = fp_generator.GetFingerprints(mols)


## Pairwise distances
**distance = 1 − similarity**. Store each pair once: `(1, 0)`, `(2, 0)`, `(2, 1)`, … . No diagonal or repeated pairs.

In [6]:
distances = []
for i in range(1, len(fps)):
    for j in range(i):
        similarity = DataStructs.TanimotoSimilarity(fps[i], fps[j])
        distances.append(1 - similarity)


## Butina clustering
Minimum center–member similarity: **0.3**. Maximum distance: **0.7**. This is a demonstration choice, not a general recommendation. The first entry of each returned cluster is its center.

In [7]:
from rdkit.ML.Cluster import Butina

similarity_cutoff = 0.3
butina_clusters = Butina.ClusterData(
    distances,
    len(fps),
    distThresh=1 - similarity_cutoff,
    isDistData=True,
    reordering=True,
)
print("Butina clusters:", butina_clusters)


Butina clusters: ((1, 2, 3, 4, 5, 7, 8, 9, 14, 15, 17), (13, 11, 12), (19, 18), (6, 0), (16,), (10,))


## Keep whole clusters together
Target molecule counts: **16 / 2 / 2**. Sort groups largest first. This allocation rule is separate from Butina.

In [8]:
def split_clusters(data, clusters):
    train_ids, val_ids, test_ids = [], [], []
    train_target = int(0.8 * len(data))
    val_target = int(0.1 * len(data))
    for cluster in sorted(clusters, key=len, reverse=True):
        if len(train_ids) + len(cluster) <= train_target:
            train_ids.extend(cluster)
        elif len(val_ids) + len(cluster) <= val_target:
            val_ids.extend(cluster)
        else:
            test_ids.extend(cluster)
    if not train_ids or not val_ids or not test_ids:
        raise ValueError("This allocation produced an empty split.")
    return (
        data.iloc[train_ids].copy(),
        data.iloc[val_ids].copy(),
        data.iloc[test_ids].copy(),
    )


In [9]:
print("Method   Train Validation Test")
for method, clusters in [
    ("scaffold", scaffold_clusters),
    ("butina", butina_clusters),
]:
    train, val, test = split_clusters(data, clusters)
    print(method, len(train), len(val), len(test))
    for name, frame in [("train", train), ("val", val), ("test", test)]:
        frame.to_csv(f"{method}_{name}.csv", index=False)


Method   Train Validation Test
scaffold 16 2 2
butina 16 2 2


## Validation
Scaffold and Butina are two separate experiments. Each exported CSV keeps only the `smiles` and `label` columns. With other data, exact target sizes may be impossible; inspect the actual sizes.

In [10]:
# Keep every row exactly once and every cluster entirely in one split.
for clusters in [scaffold_clusters, butina_clusters]:
    splits = split_clusters(data, clusters)
    split_ids = [set(frame.index) for frame in splits]
    assert [len(frame) for frame in splits] == [16, 2, 2]
    assert set.union(*split_ids) == set(range(len(data)))
    assert sum(len(ids) for ids in split_ids) == len(data)
    assert all(sum(set(c).issubset(ids) for ids in split_ids) == 1
               for c in clusters)
print("Verified: 20 rows preserved; no cluster crosses splits.")


Verified: 20 rows preserved; no cluster crosses splits.


## References
- Mobley and colleagues: [FreeSolv data and attribution](https://github.com/MobleyLab/FreeSolv).
- RDKit: [Murcko scaffolds](https://www.rdkit.org/docs/source/rdkit.Chem.Scaffolds.MurckoScaffold.html), [fingerprint generator](https://www.rdkit.org/docs/source/rdkit.Chem.rdFingerprintGenerator.html), and [Butina](https://www.rdkit.org/docs/source/rdkit.ML.Cluster.Butina.html).
